In [1]:
import torch
from tbmalt.common.maths.mixers import Simple, Anderson, _Mixer
from torchviz import make_dot

In [2]:
torch.set_default_dtype(torch.float64)
torch.set_printoptions(precision=16)

# Setup mixer

In [3]:
anderson_mixer = Anderson(False)
anderson_mixer.reset()
anderson_mixer.tolerance = 1e-15

In [4]:
def equilibrium(fcn, q0, *params):
    anderson_mixer.reset()
    max_iter = 100
    q_current = q0
    for step in range(1, max_iter + 1):
        q_new = fcn(q_current, *params)
        q_current = anderson_mixer(q_new, q_current)

        if anderson_mixer.converged:
            q_converged = q_current
            break
    return q_converged

In [5]:
def simple_func(q, x):
    return torch.sqrt(q + x)

In [6]:
def simple_func_other_jacobian(q, x):
    return torch.stack([torch.sqrt(q.sum() + x[i]) for i in range(q.shape[0])])

In [7]:
x = torch.tensor([13.1, 1.0])
x.requires_grad = True

In [8]:
with torch.no_grad():
    q = equilibrium(simple_func, torch.tensor([0.0, 2.0]), x)
q = simple_func(q, x)

q0 = q.clone().detach().requires_grad_()
f0 = simple_func(q0,x)
def backward_hook(grad):
    g = equilibrium(lambda y : torch.autograd.grad(f0, q0, y, retain_graph=True)[0] + grad,
                                       grad)
    return g
        
q.register_hook(backward_hook)

q

tensor([4.1537651812890219, 1.6180339887498949], grad_fn=<SqrtBackward0>)

In [9]:
make_dot(q).render('simple_anderson', format='png')

'simple_anderson.png'

In [10]:
torch.autograd.grad(q.sum(), x)

(tensor([0.1368451378759933, 0.4472135954999579]),)

In [25]:
testq= q.detach()
testq.requires_grad = True
testx = x.detach()
newq = simple_func(testq, testx)
newq

tensor([4.1537651812890219, 1.6180339887498949], grad_fn=<SqrtBackward0>)

In [29]:
torch.autograd.grad(newq.sum(), testq)

(tensor([0.1203727168430924, 0.3090169943749474]),)

In [17]:
torch.autograd.functional.jacobian(lambda q: simple_func(q,x), q)@torch.tensor([[1.0]
                                                                                ,[1.0]])

tensor([[0.1203727168430924],
        [0.3090169943749474]])

In [12]:
simple_func_other_jacobian(q, x)

tensor([4.3441684094932276, 2.6022680818929698], grad_fn=<StackBackward0>)

In [37]:
torch.tensor([1.0, 1.0])@torch.autograd.functional.jacobian(lambda q: simple_func_other_jacobian(q,x), q)

tensor([0.3072369080113267, 0.3072369080113267])

In [30]:
testq= q.detach()
testq.requires_grad = True
testx = x.detach()
newq = simple_func_other_jacobian(testq, testx)
newq

tensor([4.3441684094932276, 2.6022680818929698], grad_fn=<StackBackward0>)

In [32]:
torch.autograd.grad(newq.sum(), testq)

(tensor([0.3072369080113267, 0.3072369080113267]),)